# Data Science Blog Post: Developer Salaries & Experience

## 📊 Project Overview
This project analyzes the 2024 Stack Overflow Developer Survey to answer questions about developer salaries, experience, education, and job satisfaction.

## ❓ Business Questions
1. Does education level (EdLevel) affect salary?
2. Does education level affect years of experience?
3. Does experience affect job satisfaction?
4. Does experience affect salary?

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score , root_mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns


## 2. Data Understanding
Loading the dataset and exploring its structure.

In [ ]:
df = pd.read_csv('stack-overflow-developer-survey-2024/survey_results_public.csv')
df.head()

In [ ]:
df.shape

In [ ]:
df.columns.tolist()

## 3. Data Preparation
Cleaning salary and experience columns, handling missing values and outliers.

In [ ]:
missing_salary_count=df['ConvertedCompYearly'].isnull().sum()
print(missing_salary_count)

In [ ]:
df_salary = df.dropna(subset=['ConvertedCompYearly'])
df_salary.shape

In [ ]:
df_salary['ConvertedCompYearly'].describe()

In [ ]:
df_clean = df_salary[(df_salary['ConvertedCompYearly'] >= 12000) & (df_salary['ConvertedCompYearly'] <= 600000)].copy() 
df_clean.shape

**Why these salary cutoffs?**

The raw data contained unrealistic values — a minimum of $1 and a maximum of $16 million — 
which are almost certainly data entry errors, jokes, or misunderstood questions. I chose to 
keep salaries between **$12,000 and $600,000** for these reasons:

- **$12,000 lower bound**: removes obvious junk values ($1, etc.) while preserving junior 
  developers in low-cost countries where low salaries are legitimate.
- **$600,000 upper bound**: removes fantasy values ($16M) while keeping real high-earning 
  senior engineers at top tech companies (e.g., FAANG).
- **Validation**: I visually inspected the scatter plot after filtering. There is no dense 
  cluster of points glued to the $600K ceiling, which suggests the cap removed outliers 
  without censoring real high earners.

In [ ]:
df_clean['YearsCodePro'] = df_clean['YearsCodePro'].replace({
    'Less than 1 year': 0.5,
    'More than 50 years': 55
}) 

**Why these conversions for YearsCodePro?**

The YearsCodePro column mixed numeric values with two text values. I converted them to 
numbers so the column could be used in linear regression:

- **"Less than 1 year" → 0.5**: The midpoint is more accurate than 0, since these developers 
  do have *some* experience — just less than a full year.
- **"More than 50 years" → 55**: A reasonable number slightly above 50. Only 50 people fall 
  into this category, so the exact value has a minimal impact on the analysis.

After replacement, I used `pd.to_numeric()` to convert the column from object (string) type 
to float64, which is required for arithmetic operations and model fitting.

In [ ]:
df_clean['YearsCodePro'] = pd.to_numeric(df_clean['YearsCodePro']) 

In [ ]:
df_salary_exp = df_clean.dropna(subset=['YearsCodePro'])
df_salary_exp.shape

In [ ]:
df_clean['JobSat'].isnull().sum()

In [ ]:
df_jobsat = df_clean.dropna(subset=['JobSat', 'YearsCodePro']).copy()
df_jobsat.shape

In [ ]:
edu_rank = {
    'Primary/elementary school': 1,
    'Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)': 2,
    'Some college/university study without earning a degree': 3,
    'Associate degree (A.A., A.S., etc.)': 4,
    'Bachelor’s degree (B.A., B.S., B.Eng., etc.)': 5,
    'Master’s degree (M.A., M.S., M.Eng., MBA, etc.)': 6,
    'Professional degree (JD, MD, Ph.D, Ed.D, etc.)': 7
}

In [ ]:
df_clean['EdLevel_rank'] = df_clean['EdLevel'].map(edu_rank)

**Why ordinal encoding for EdLevel?**

Education levels have a **natural order** (Primary < Secondary < Bachelor's < Master's < PhD), 
so ordinal encoding is more appropriate than one-hot encoding. This treats education as a 
single numeric feature where higher values represent more education, allowing the regression 
model to capture a linear trend if one exists.

**Assumptions and limitations:**
- I ranked "Some college without a degree" between Secondary (2) and Associate (4) as level 3, 
  treating incomplete college as partial progress beyond high school.
- The survey groups JD, MD, and PhD together as "Professional degree," so I treated that tier 
  as the highest (7), above Master's.
- **"Something else" was dropped** (932 rows) because it doesn't fit the education hierarchy, 
  and ranking it would require making up a value with no clear meaning.

In [ ]:
df_edu = df_clean.dropna(subset=['EdLevel_rank', 'YearsCodePro']).copy()
df_edu.shape

In [ ]:
df_q1 = df_clean.dropna(subset=['EdLevel_rank']).copy()
df_q1.shape

## 4. Reusable Function

In [ ]:
def run_regression(df, feature_col, target_col):
    """
    Runs a simple linear regression and prints MSE, RMSE, and R².
    
    Parameters:
    - df: the DataFrame to use
    - feature_col: the column name to use as input (X)
    - target_col: the column name to use as output (y)
    """
    X = df[[feature_col]]
    y = df[target_col]
    
    lr = LinearRegression()
    lr.fit(X, y)
    predictions = lr.predict(X)
    
    mse = mean_squared_error(y, predictions)
    rmse = root_mean_squared_error(y, predictions)
    r2 = r2_score(y, predictions)
    
    print(f"Feature: {feature_col}  →  Target: {target_col}")
    print(f"RMSE: {rmse:,.2f}")
    print(f"MSE:  {mse:,.2f}")
    print(f"R²:   {r2:.4f}")

## 5. Analysis

### Q1: Does education level affect salary?

In [ ]:
run_regression(df_q1, 'EdLevel_rank', 'ConvertedCompYearly')

In [ ]:
plt.figure(figsize=(10, 6))
df_q1.boxplot(column='ConvertedCompYearly', by='EdLevel_rank')
plt.xlabel('Education Level (1=Primary → 7=Professional)')
plt.ylabel('Salary (USD)')
plt.title('Salary by Education Level')
plt.suptitle('')
plt.show()

**Answer:** Education level has almost no linear effect on salary (R² = 0.005). 
While medians are roughly flat across education levels, the boxplot reveals a 
subtle insight: higher education increases the *chance* of reaching top salaries 
($300K+), but doesn't guarantee a higher typical salary. A PhD won't guarantee 
more money, but it widens the ceiling of possibilities.

### Q2: Does education level affect years of experience?

In [ ]:
run_regression(df_edu, 'EdLevel_rank', 'YearsCodePro')

In [ ]:
plt.figure(figsize=(10, 6))
df_edu.boxplot(column='YearsCodePro', by='EdLevel_rank')
plt.xlabel('Education Level (1=Primary → 7=Professional)')
plt.ylabel('Years of Professional Experience')
plt.title('Years of Experience by Education Level')
plt.suptitle('')
plt.show()


**Answer:** Education level has essentially zero effect on years of experience 
(R² = 0.0008). The boxplot shows that developers at every education level span 
the full range of experience — from junior to veteran. Why? Because experience 
depends on *age and when you started coding*, not how much school you attended. 
A self-taught dev who started at 16 can easily have more experience at 40 than 
a PhD holder who started coding in grad school.


### Q3: Does experience affect job satisfaction?

In [ ]:
run_regression(df_jobsat, 'YearsCodePro', 'JobSat')

In [ ]:
# Scatter plot (note: limited visibility due to JobSat having only 11 discrete values)
plt.figure(figsize=(10, 6))
plt.scatter(df_jobsat['YearsCodePro'], df_jobsat['JobSat'], alpha=0.3)
plt.xlabel('Years of Professional Experience')
plt.ylabel('Job Satisfaction (0-10)')
plt.title('Job Satisfaction vs Years of Experience')
plt.show()

In [ ]:
# Boxplot with experience buckets for clearer view
df_jobsat['ExpBucket'] = pd.cut(
    df_jobsat['YearsCodePro'],
    bins=[0, 5, 15, 30, 60],
    labels=['0-5 years', '5-15 years', '15-30 years', '30+ years']
)

plt.figure(figsize=(10, 6))
df_jobsat.boxplot(column='JobSat', by='ExpBucket')
plt.xlabel('Experience Level')
plt.ylabel('Job Satisfaction')
plt.title('Job Satisfaction by Experience Level')
plt.suptitle('')
plt.show()

**Answer:** Years of experience has almost no effect on job satisfaction 
(R² = 0.008). While developers (30+ years) show a slight upward trend 
in satisfaction, the overlap between groups is huge — developers at every 
experience level report the full range of satisfaction scores (0–10). 
Happiness at work is driven by other factors: company culture, salary, 
work-life balance, and type of work likely matter far more than years alone.

### Q4: Does experience affect salary?

In [ ]:
run_regression(df_salary_exp, 'YearsCodePro', 'ConvertedCompYearly')

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(df_salary_exp['YearsCodePro'], df_salary_exp['ConvertedCompYearly'], alpha=0.3)
plt.xlabel('Years of Professional Experience')
plt.ylabel('Salary (USD)')
plt.title('Salary vs Years of Experience')
plt.show()

In [ ]:
# Check correlation
df_salary_exp['YearsCodePro'].corr(df_salary_exp['ConvertedCompYearly'])
# Q4 Finding: Correlation between YearsCodePro and salary = 0.333
# This is a weak-to-moderate positive relationship.
# R² = 0.11 → experience alone explains only ~11% of salary variation.
# Conclusion: Experience matters, but many other factors also drive salary.

**Answer:** Experience is the strongest predictor among our features, but 
still only weak (R² = 0.111, correlation = 0.333). On average, the model's 
predictions are off by ~$63,000 — roughly the size of a median salary itself. 
Salary rises with experience early on, then plateaus after ~10-15 years. 
Experience matters, but many other factors (country, company, negotiation, 
language specialization) clearly play a much larger role in determining pay.

## 6. Overall Conclusions

| Question | R² | Finding |
|----------|-----|---------|
| Q1: Education → Salary | 0.005 | Almost no effect (but ceiling effect visible) |
| Q2: Education → Experience | 0.0008 | Zero effect |
| Q3: Experience → Satisfaction | 0.008 | Almost no effect |
| Q4: Experience → Salary | 0.111 | Weak positive |

### Key Takeaway
Developer careers are complicated. Neither experience nor education alone can 
reliably predict salary, happiness, or each other. The factors that really 
drive developer outcomes are likely things NOT easily captured in a survey — 
the specific company, country, negotiation skills, network, and luck. 
Experience is the best single predictor of salary we found, but it still 
explains only ~11% of the variation. The story isn't *"do this one thing to 
succeed"* — it's *"success depends on many things working together."*